<a href="https://colab.research.google.com/github/AntonDozhdikov/AntonDozhdikov/blob/main/Qwen2_5_7B_Instruct_generation_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Установка необходимых библиотек
!pip install -q fsspec==2025.3.2 transformers datasets peft torch pandas openpyxl trl bitsandbytes accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.3/519.3 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 123.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 100.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 104.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# Импорт необходимых модулей
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import Dataset
from google.colab import files
import gc
import time  # Для измерения времени обучения

In [ ]:
# Проверка доступности GPU
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU is available.")
else:
    device = torch.device("cpu")
    print("GPU not available, using CPU.")

GPU is available.


In [ ]:
# Загрузка датасета
uploaded = files.upload()  # Загрузите sinema5.xlsx через диалог
df = pd.read_excel('sinema5.xlsx')

Saving sinema5.xlsx to sinema5.xlsx


In [ ]:
# Анализ структуры и баланса классов
print(f"Всего фильмов: {len(df)}")
print(f"Успешных фильмов: {df['rezult2'].sum()}")
print(f"Неуспешных фильмов: {len(df) - df['rezult2'].sum()}")


Всего фильмов: 1683
Успешных фильмов: 180
Неуспешных фильмов: 1503


In [ ]:
# Проверка типов данных и их корректировка (важно!)
for col in df.columns:
    if df[col].dtype == 'object':  # Проверяем, является ли столбец строковым
        df[col] = df[col].astype(str)  # Явно преобразуем в строку
    elif df[col].dtype == 'int64':  # Проверяем, является ли столбец int64
        pass # Оставляем как есть, если это ID или результат

# Явная проверка rezult2 (очень важно)
df['rezult2'] = df['rezult2'].fillna(0).astype(int)


In [ ]:
# Заполнение пропусков
df['tagline'] = df['tagline'].fillna('')
df['logline'] = df['logline'].fillna('')
df['annotation'] = df['annotation'].fillna('')
df['genre'] = df['genre'].fillna('')
df['name'] = df['name'].fillna('')

In [ ]:
# Функция для создания текстового промпта
def create_prompt(name, genre, logline, task_type):
    if task_type == "tagline":
        prompt = f"""Ты — опытный копирайтер, создающий слоганы для фильмов.
Название фильма: {name}
Жанр: {genre}
Краткое описание: {logline}
Сгенерируй яркий и запоминающийся тэглайн, подчеркивающий коммерческий потенциал фильма."""
    elif task_type == "annotation":
        prompt = f"""Ты — профессиональный сценарист. Напиши подробную аннотацию (синопсис) для фильма.
Название фильма: {name}
Жанр: {genre}
Краткое описание: {logline}
Аннотация должна заинтересовать зрителя и подчеркнуть потенциал кассового успеха."""
    else:
        raise ValueError("Неизвестный task_type. Используйте 'tagline' или 'annotation'.")
    return prompt

In [ ]:
# Функция форматирования данных для обучения
def formatting_func(example):
    return f"<|im_start|>user\n{example['input']}<|im_end|>\n<|im_start|>assistant\n{example['output']}<|im_end|>"


In [ ]:
# Формирование обучающих данных
data = []
for _, row in df.iterrows():
    # Добавляем пример для генерации тэглайна
    prompt_tagline = create_prompt(row['name'], row['genre'], row['logline'], "tagline")
    data.append({"input": prompt_tagline, "output": row['tagline']})

    # Добавляем пример для генерации аннотации
    prompt_annot = create_prompt(row['name'], row['genre'], row['logline'], "annotation")
    data.append({"input": prompt_annot, "output": row['annotation']})

# Применяем функцию форматирования и добавляем столбец "text"
formatted_data = []
for item in data:
    item["text"] = formatting_func(item)
    formatted_data.append(item)


In [ ]:
# Преобразование данных в Dataset Hugging Face (после добавления поля "text")
dataset = Dataset.from_pandas(pd.DataFrame(formatted_data))

# Разделение на обучающую и валидационную выборки
dataset = dataset.train_test_split(test_size=0.2, seed=42)

In [ ]:
# Загрузка модели и токенизатора
model_id = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [ ]:
# Квантизация модели
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [ ]:
# Подготовка модели для LoRA
model = prepare_model_for_kbit_training(model)

# Конфигурация LoRA
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)
model = get_peft_model(model, peft_config)

In [ ]:
# Параметры обучения
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    optim="paged_adamw_32bit",
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=True,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    report_to="none",
    logging_steps=10,
    save_strategy="epoch",
)

In [ ]:
# Инициализация SFTTrainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,  # !!! Важно: Явно передаем токенизатор
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    dataset_text_field="text" # !!! Важно: Указываем имя поля с текстом
)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.11/dist-packages/trl/trainer/sft_trainer.py:292: UserWarning: You didn't pass a `max_seq_length` argument to the SFTTrainer, this will default to 1024
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/trl/trainer/sft_trainer.py:321: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/2692 [00:00<?, ? examples/s]

Map:   0%|          | 0/674 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/trl/trainer/sft_trainer.py:401: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  super().__init__(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [ ]:
# Обучение модели
start_time = time.time()
trainer.train()
training_time = time.time() - start_time
print(f"Training completed in {training_time:.2f} seconds")

Step,Training Loss
10,2.301900
20,2.114800
30,1.819800
40,1.490900
50,1.242700
60,1.122000
70,1.092800
80,1.008700
90,1.096100
100,1.146500


Training completed in 6404.52 seconds


In [ ]:
# Функция для генерации текста (tagline или annotation)
def generate_text(name, genre, logline, task_type):
    prompt = create_prompt(name, genre, logline, task_type)
    inputs = tokenizer(f"<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n", return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=512 if task_type == "annotation" else 32,  # Длина ответа зависит от задачи
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            eos_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(output[0], skip_special_tokens=False)
    response = response.split("<|im_start|>assistant\n")[-1].split("<|im_end|>")[0].strip()
    return response


In [ ]:
# Тестирование модели
test_name = "Двойной апгрейд"
test_genre = "комедия, фантастика, боевик"
test_logline = "Борзая киборгиня Угар уводит у актрисы-натуралки по имени Трэш именитого режиссера фильма про криминальные 90-ые годы  - Танцор техно. Для победы над чипованой соперницей, которая сгодилась бы ей в дочери, Трэш подвергает свое тело апгрейду. Кто из актрис выключит человечность в битве за мужчину и главную роль?."
print("Тестирование модели:")
print(f"Тэглайн: {generate_text(test_name, test_genre, test_logline, 'tagline')}")
print(f"Аннотация: {generate_text(test_name, test_genre, test_logline, 'annotation')}")


Тестирование модели:
Тэглайн: Все ради любви! Всё ради事业助手：0
0
Аннотация: История о том, как настоящая женщина может стать настоящей героиней. События разворачиваются в мировом киноцентре, где встречаются две актрисы — Трэш и Угар. Обе мечтают стать главной героиней следующего блокбастера о криминальных 90-х, но только Угар уже заключила контракт на роль, а Трэш пока не имеет ничего, кроме искренних намерений. В желание Трэш забрать роль у Угар получается неожиданный и экстремальный способ: она подвергает свое тело апгрейду. Теперь Трэш — человек, который может все. Но так ли все просто? И может ли она действительно стать такой, какой ей нужна? Пока Угар, используя свои киборгские способности, решает отобрать у Трэш главную роль, Трэш понимает, что если хочет выжить, то необходимо победить не только в командной игре, но и в личной войне против собственного тела и своих чувств. И только теперь она понимает, что самое важное — это быть собой. И своим телом. И своими чувствами. И своими о

In [ ]:
# Сохранение модели и токенизатора
gc.collect()
torch.cuda.empty_cache()
output_dir = "qwen2.5-7b-lora-film"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
!zip -r {output_dir}.zip {output_dir}
files.download(f"{output_dir}.zip")

print("Модель сохранена и готова к скачиванию.")

  adding: qwen2.5-7b-lora-film/ (stored 0%)
  adding: qwen2.5-7b-lora-film/tokenizer_config.json (deflated 83%)
  adding: qwen2.5-7b-lora-film/added_tokens.json (deflated 67%)
  adding: qwen2.5-7b-lora-film/special_tokens_map.json (deflated 63%)
  adding: qwen2.5-7b-lora-film/merges.txt (deflated 57%)
  adding: qwen2.5-7b-lora-film/adapter_config.json (deflated 54%)
  adding: qwen2.5-7b-lora-film/vocab.json (deflated 61%)
  adding: qwen2.5-7b-lora-film/README.md (deflated 66%)
  adding: qwen2.5-7b-lora-film/adapter_model.safetensors (deflated 8%)
  adding: qwen2.5-7b-lora-film/tokenizer.json (deflated 81%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Модель сохранена и готова к скачиванию.
